# TF-IDF on Unstructured Text (NLP tools only, no DataFrames)

All steps of `task_to_do.txt` are done with **scikit-learn's NLP text tools**
(`TfidfVectorizer`, `cosine_similarity`) plus **NumPy / SciPy sparse matrices**.
`pandas` is deliberately **not** used: the TF-IDF matrix stays a sparse
`scipy.sparse.csr_matrix` and is inspected through the vectorizer's vocabulary
(`get_feature_names_out()`) and NumPy indexing.

Problems met while doing this are logged in the `issues_faced/` folder.

## Step 1 — Load the data (10 documents)

In [1]:
import os, sys

# Notebook lives in notebooks/ -> put the repo root on sys.path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(REPO_ROOT, "data")):
    REPO_ROOT = os.path.abspath(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.load_data import load_documents

titles, documents = load_documents()
assert len(documents) == 10, "expected 10 documents"
print(f"{len(documents)} documents loaded.\n")
print(f"{'Title':<25s} Words")
print("-" * 32)
for t, d in zip(titles, documents):
    print(f"{t:<25s} {len(d.split()):>5d}")

10 documents loaded.

Title                     Words
--------------------------------
AI in Healthcare             85
Climate Change               83
Basketball Strategy          85
Italian Pasta                91
Space Exploration            86
Cybersecurity                88
Music Theory                 87
Gardening Vegetables         89
Economics Inflation          89
Public Health                91


## Step 2 — Build the TF-IDF matrix
`TfidfVectorizer(stop_words="english")` tokenises, lower-cases, removes English
stop words and weights terms by TF-IDF in a single step.

In [2]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(documents)          # scipy sparse CSR matrix
terms = vectorizer.get_feature_names_out()

print("Matrix type    :", type(X).__name__)
print("Matrix shape   :", X.shape, " (10, n_unique_terms)")
print("Vocabulary size:", len(terms))
print(f"Non-zero cells : {X.nnz} of {X.shape[0] * X.shape[1]} "
      f"({100 * X.nnz / (X.shape[0] * X.shape[1]):.1f}% dense)")
assert X.shape == (10, len(terms))

Matrix type    : csr_matrix
Matrix shape   : (10, 537)  (10, n_unique_terms)
Vocabulary size: 537
Non-zero cells : 620 of 5370 (11.5% dense)


## Step 3 — Inspect the matrix (without a DataFrame)
Slice the sparse matrix, densify only the small slice, and print it using the
vocabulary as column labels.

In [3]:
def print_slice(matrix, row_labels, col_labels, n_rows=3, n_cols=10, width=11):
    """Pretty-print the top-left corner of a sparse matrix with labels."""
    block = matrix[:n_rows, :n_cols].toarray()
    print(" " * 22 + "".join(f"{c[:width - 1]:>{width}s}" for c in col_labels[:n_cols]))
    for label, row in zip(row_labels[:n_rows], block):
        print(f"{label:<22s}" + "".join(f"{v:>{width}.3f}" for v in row))

print("Sample of the TF-IDF matrix (first 3 docs x 10 terms):\n")
print_slice(X, titles, terms, 3, 10)

Sample of the TF-IDF matrix (first 3 docs x 10 terms):

                          ability accelerate     access accountabi   accuracy   accurate    actions   activity adaptation        add
AI in Healthcare            0.000      0.000      0.000      0.125      0.125      0.000      0.000      0.000      0.000      0.000
Climate Change              0.000      0.000      0.000      0.000      0.000      0.125      0.125      0.000      0.125      0.000
Basketball Strategy         0.130      0.000      0.000      0.000      0.000      0.000      0.000      0.000      0.000      0.000


In [4]:
def top_terms(matrix, col_labels, row_idx, n=5):
    """Top-n (term, score) for one document, read from the sparse CSR row."""
    row = matrix.getrow(row_idx)
    order = np.argsort(row.data)[::-1][:n]
    return [(col_labels[row.indices[i]], row.data[i]) for i in order]

print("Top 5 TF-IDF terms per document")
print("=" * 40)
for i, t in enumerate(titles):
    print(f"\n{t}:")
    for term, score in top_terms(X, terms, i, 5):
        print(f"  {term:<20s} {score:.3f}")

Top 5 TF-IDF terms per document

AI in Healthcare:
  ai                   0.374
  healthcare           0.249
  use                  0.165
  waiting              0.125
  scan                 0.125

Climate Change:
  emissions            0.250
  change               0.250
  energy               0.212
  climate              0.212
  communities          0.125

Basketball Strategy:
  situations           0.130
  pick                 0.130
  game                 0.130
  selection            0.130
  shot                 0.130

Italian Pasta:
  pasta                0.443
  sauces               0.332
  sauce                0.222
  different            0.188
  timer                0.111

Space Exploration:
  space                0.349
  missions             0.233
  mars                 0.233
  science              0.116
  telescopes           0.116

Cybersecurity:
  cybersecurity        0.249
  systems              0.212
  data                 0.165
  ransomware           0.125
  firewalls      

## Step 4 — Query similarity challenge
The query is transformed with the **same fitted vectorizer** (so it shares the
vocabulary and IDF weights) and compared with every document by cosine similarity.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def rank_documents(query, vec=None, matrix=None, show=True):
    """Return [(title, score)] sorted by cosine similarity (descending)."""
    vec = vec if vec is not None else vectorizer
    matrix = matrix if matrix is not None else X
    q = vec.transform([query])
    scores = cosine_similarity(q, matrix).ravel()
    ranking = sorted(zip(titles, scores), key=lambda p: p[1], reverse=True)
    if show:
        print(f"\n=== Query: {query!r} ===")
        for rank, (title, score) in enumerate(ranking, 1):
            print(f"  {rank:>2d}. {title:<25s} {score:.3f}")
    return ranking

queries = {
    "renewable energy and climate policy": "Climate Change",
    "password security and phishing":      "Cybersecurity",
    "healthy sleep exercise and diet":     "Public Health",
}

for q, expected in queries.items():
    ranking = rank_documents(q)
    status = "PASS" if ranking[0][0] == expected else "FAIL"
    print(f"  -> expected top: {expected!r}  [{status}]")


=== Query: 'renewable energy and climate policy' ===
   1. Climate Change            0.323
   2. Economics Inflation       0.095
   3. Gardening Vegetables      0.048
   4. AI in Healthcare          0.000
   5. Basketball Strategy       0.000
   6. Italian Pasta             0.000
   7. Space Exploration         0.000
   8. Cybersecurity             0.000
   9. Music Theory              0.000
  10. Public Health             0.000
  -> expected top: 'Climate Change'  [PASS]

=== Query: 'password security and phishing' ===
   1. Cybersecurity             0.176
   2. AI in Healthcare          0.000
   3. Climate Change            0.000
   4. Basketball Strategy       0.000
   5. Italian Pasta             0.000
   6. Space Exploration         0.000
   7. Music Theory              0.000
   8. Gardening Vegetables      0.000
   9. Economics Inflation       0.000
  10. Public Health             0.000
  -> expected top: 'Cybersecurity'  [PASS]

=== Query: 'healthy sleep exercise and diet' ===


**Observations.** Each query contains words that are rare across the corpus
(*renewable*, *phishing*, *password*, *sleep*, *diet* ...), so they carry a high
IDF and pull the matching document to rank 1. Words in the query that are not in
the vocabulary (or are stop words, e.g. *and*) are simply ignored.

## Step 5 — Optional challenge: bigrams, `max_df`, `min_df`

In [6]:
vectorizer_v2 = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2,
)
X_v2 = vectorizer_v2.fit_transform(documents)
terms_v2 = vectorizer_v2.get_feature_names_out()

# Reference: bigrams only, with no document-frequency filtering
vectorizer_v3 = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
X_v3 = vectorizer_v3.fit_transform(documents)

print(f"{'Setting':<42s} {'Shape':<12s} Vocab")
print("-" * 66)
print(f"{'v1: unigrams (baseline)':<42s} {str(X.shape):<12s} {len(terms)}")
print(f"{'v3: (1,2)-grams, no df filter':<42s} {str(X_v3.shape):<12s} {len(vectorizer_v3.get_feature_names_out())}")
print(f"{'v2: (1,2)-grams, max_df=0.8, min_df=2':<42s} {str(X_v2.shape):<12s} {len(terms_v2)}")

print("\nBigrams kept in v2:", [t for t in terms_v2 if " " in t])
print("\nTerms kept in v2  :", list(terms_v2))

Setting                                    Shape        Vocab
------------------------------------------------------------------
v1: unigrams (baseline)                    (10, 537)    537
v3: (1,2)-grams, no df filter              (10, 1182)   1182
v2: (1,2)-grams, max_df=0.8, min_df=2      (10, 69)     69

Bigrams kept in v2: ['economic growth', 'policy involves', 'trade offs']

Terms kept in v2  : ['adjust', 'based', 'careful', 'clean', 'climate', 'communication', 'data', 'decisions', 'defenses', 'depends', 'design', 'different', 'disease', 'economic', 'economic growth', 'efficiency', 'energy', 'essential', 'flavor', 'global', 'governments', 'growth', 'health', 'healthy', 'help', 'home', 'improve', 'improves', 'individual', 'influence', 'intelligence', 'international', 'involves', 'large', 'learning', 'live', 'local', 'long', 'medical', 'monitoring', 'nutrition', 'offs', 'personal', 'plans', 'policies', 'policy', 'policy involves', 'populations', 'practice', 'privacy', 'quality', 'r

In [7]:
print("Top 5 terms per document: v1 (unigrams) vs v2 (bigrams, df-filtered)\n")
for i, t in enumerate(titles):
    v1 = ", ".join(f"{w}" for w, _ in top_terms(X, terms, i, 5))
    v2 = ", ".join(f"{w}" for w, _ in top_terms(X_v2, terms_v2, i, 5)) or "(no terms left)"
    print(f"{t}\n  v1: {v1}\n  v2: {v2}\n")

empty = [t for t, n in zip(titles, np.diff(X_v2.indptr)) if n == 0]
print("Documents with NO terms left in v2:", empty or "none")
print("Non-zero terms per document in v2:",
      dict(zip(titles, map(int, np.diff(X_v2.indptr)))))

Top 5 terms per document: v1 (unigrams) vs v2 (bigrams, df-filtered)

AI in Healthcare
  v1: ai, healthcare, use, waiting, scan
  v2: use, populations, support, privacy, careful

Climate Change
  v1: emissions, change, energy, climate, communities
  v2: climate, energy, economic growth, trade offs, large

Basketball Strategy
  v1: situations, pick, game, selection, shot
  v2: based, adjust, practice, successful, communication

Italian Pasta
  v1: pasta, sauces, sauce, different, timer
  v2: different, long, flavor, tomatoes, quality

Space Exploration
  v1: space, missions, mars, science, telescopes
  v2: international, medical, health, water, data

Cybersecurity
  v1: cybersecurity, systems, data, ransomware, firewalls
  v2: systems, data, training, risk, privacy

Music Theory
  v1: music, musicians, practice, memorable, perform
  v2: practice, improves, live, training, software

Gardening Vegetables
  v1: soil, year, pests, improves, crops
  v2: improves, local, nutrition, careful, c

In [8]:
print("Re-running the three queries with the v2 vectorizer:")
for q, expected in queries.items():
    ranking = rank_documents(q, vectorizer_v2, X_v2, show=False)
    top, score = ranking[0]
    note = "  <- no query term in vocabulary: ranking is an arbitrary tie" if score == 0 else ""
    print(f"  {q!r:<40s} top = {top!r:<20s} score = {score:.3f} "
          f"(expected {expected!r}: {'PASS' if top == expected else 'FAIL'}){note}")

Re-running the three queries with the v2 vectorizer:
  'renewable energy and climate policy'    top = 'Climate Change'     score = 0.557 (expected 'Climate Change': PASS)
  'password security and phishing'         top = 'AI in Healthcare'   score = 0.000 (expected 'Cybersecurity': FAIL)  <- no query term in vocabulary: ranking is an arbitrary tie
  'healthy sleep exercise and diet'        top = 'Gardening Vegetables' score = 0.218 (expected 'Public Health': FAIL)


### Does v2 make the terms more meaningful?
**No - not on this corpus.**

* Vocabulary: 537 (unigrams) -> 1182 (bigrams, no filtering) -> only **69** with
  `max_df=0.8, min_df=2`.
* With only 10 documents, `min_df=2` deletes every term that is unique to one
  document - exactly the discriminative vocabulary (*phishing*, *pasta*, *rovers*,
  *ransomware* ...). What is left is generic (*use*, *help*, *time*, *different*).
* Only 3 bigrams survive (`economic growth`, `policy involves`, `trade offs`) and all
  come from the Climate/Economics documents, so most documents get no bigrams.
* Top terms for AI in Healthcare, Basketball, Space, Cybersecurity, Music etc. became
  **less** topical (e.g. Basketball -> *based, adjust, practice*).
* Query effect: the Climate query still works (0.557), but **"password security and
  phishing" and "healthy sleep exercise and diet" now FAIL** - their key words were
  filtered out, every score is 0 (or tiny) and the "winner" is just a tie.
* Bigrams / `min_df` / `max_df` pay off on a *large* corpus (e.g. *machine learning*,
  removing typos), not on 10 short documents. See `issues_faced/` for details.

## Step 6 — Verify with the unit tests (`pytest -q`)

In [9]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_ROOT,
                        capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-500:])
assert result.returncode == 0, "pytest failed"

...                                                                      [100%]
3 passed in 3.08s




## Exit ticket

1. **What does a high TF-IDF score mean for a term in one document?**
   The term appears often in that document (high TF) *and* is rare across the other
   documents (high IDF) - so it is a good identifier of what that document is about.
2. **Why does IDF reduce the weight of common words like "the" or "data"?**
   IDF = log(N / df) (smoothed). A word that occurs in most documents has df close
   to N, so IDF is close to its minimum and the word cannot distinguish documents.
3. **Name one real task where TF-IDF is useful.**
   Search / document retrieval (ranking pages for a query), keyword extraction,
   and as a baseline feature set for text classification or spam filtering.